In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import mteb
from mteb.benchmarks import get_benchmark

In [10]:
VN_MTEB_BENCHMARK = 'VN-MTEB (vie, v1)'
vn_benchmark = mteb.benchmarks.get_benchmark(VN_MTEB_BENCHMARK)
all_tasks = list(vn_benchmark.tasks)

task_groups = {
    "STS":                [t for t in all_tasks if t.metadata.type == "STS"],
    "Retrieval":          [t for t in all_tasks if t.metadata.type == "Retrieval"],
    "Reranking":          [t for t in all_tasks if t.metadata.type == "Reranking"],
    "Classification":     [t for t in all_tasks if t.metadata.type == "Classification"],
    "Clustering":         [t for t in all_tasks if t.metadata.type == "Clustering"],
    "PairClassification": [t for t in all_tasks if t.metadata.type == "PairClassification"],
}

for name, tasks in task_groups.items():
    print(f"{name}: {len(tasks)} tasks")

STS: 3 tasks
Retrieval: 24 tasks
Reranking: 3 tasks
Classification: 12 tasks
Clustering: 5 tasks
PairClassification: 3 tasks


In [16]:
task_groups["Retrieval"]
TASK_BLACKLIST = ["CQADupstackWebmasters-VN"]  # e.g. ["BIOSSES-VN", "SICK-R-VN"]

RUN_TASKS = [t for t in task_groups["Retrieval"] if t.metadata.name in TASK_BLACKLIST]
RUN_TASKS

[CQADupstackWebmastersVN(name='CQADupstackWebmasters-VN', languages=['vie'])]

In [ ]:
import os
import torch
from sentence_transformers import SentenceTransformer
from huggingface_hub import snapshot_download

MODEL_NAMES = [
    # 'google/embeddinggemma-300m',
    os.path.abspath('../model/gemma300-update-reindex'),
    # 'quockhangdev/vilegal'
]

# VILEGAL_SUBFOLDER = "embeddinggemma-300m-vilegal-stage2-hardneg-trimmed"
# TASK_GROUP = "STS"
# TASK_BLACKLIST = []  # e.g. ["BIOSSES-VN", "SICK-R-VN"]

# RUN_TASKS = [t for t in task_groups[TASK_GROUP] if t.metadata.name in TASK_BLACKLIST]

# def load_model(model_name):
#     if model_name == 'quockhangdev/vilegal':
#         # only download the subfolder, not the whole repo
#         local_path = snapshot_download(
#             model_name,
#             allow_patterns=f"{VILEGAL_SUBFOLDER}/*"
#         )
#         return SentenceTransformer(os.path.join(local_path, VILEGAL_SUBFOLDER))
#     return SentenceTransformer(model_name)

all_results = {}
for model_name in MODEL_NAMES:
    print(f"\n=== Evaluating: {model_name} ===")
    model = SentenceTransformer(model_name)
    all_results[model_name] = mteb.evaluate(
        model=model,
        tasks=RUN_TASKS,
        encode_kwargs={"batch_size": 16, 'show_progress_bar': True},
        show_progress_bar=True,
        num_proc=4,
        cache=mteb.cache.ResultCache("../vn_mteb_results"),
    )


=== Evaluating: /Users/minh/Code/Project/lm-vocab-trimmer/model/gemma300-update-reindex ===


Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

/Users/minh/Code/Project/lm-vocab-trimmer/venv/lib/python3.14/site-packages/mteb/models/model_meta.py:681: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimensions = model.get_sentence_embedding_dimension()
/Users/minh/Code/Project/lm-vocab-trimmer/venv/lib/python3.14/site-packages/mteb/models/model_meta.py:646: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embed_dim=model.get_sentence_embedding_dimension(),


Evaluating tasks: 0it [00:00, ?it/s]

UnboundLocalError: cannot access local variable '_res' where it is not associated with a value

In [ ]:
import os
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAMES = [

    # 'google/embeddinggemma-300m',
    os.path.abspath('../model/gemma300-vi-trimmed'),
]

RUN_TASKS = [mteb.get_task("ZacLegalTextRetrieval")]

all_results = {}
for model_name in MODEL_NAMES:
    print(f"\n=== Evaluating: {model_name} ===")
    model = SentenceTransformer(model_name)
    all_results[model_name] = mteb.evaluate(
        model=model,
        tasks=RUN_TASKS,
        encode_kwargs={
            "batch_size": 64,
            "show_progress_bar": True,
            "normalize_embeddings": True,
        },
        show_progress_bar=True,
        num_proc=4,
        cache=mteb.cache.ResultCache("../vn_mteb_results"),
    )


=== Evaluating: /Users/minh/Code/Project/lm-vocab-trimmer/model/gemma300-vi-trimmed ===


FileNotFoundError: Path /Users/minh/Code/Project/lm-vocab-trimmer/model/gemma300-vi-trimmed not found

In [ ]:
import pandas as pd

labels = list(all_results.keys())
rows = []
for model_name, results in all_results.items():
    df = results.to_dataframe()
    score_col = df.columns[-1]
    for _, row in df.iterrows():
        rows.append({'model': model_name, 'task': row['task_name'], 'score': row[score_col]})

df = pd.DataFrame(rows).pivot(index='task', columns='model', values='score')

# rename columns to short labels
col_map = {labels[0]: 'original', labels[-1]: 'trimmed'}
df = df.rename(columns=col_map)

if 'trimmed' in df.columns and 'original' in df.columns:
    df['diff'] = df['trimmed'] - df['original']

print(df.to_string())

model             trimmed  original      diff
task                                         
BIOSSES-VN       0.818043  0.783512  0.034531
SICK-R-VN        0.788102  0.788126 -0.000024
STSBenchmark-VN  0.808265  0.819424 -0.011159


In [ ]:
from transformers import AutoTokenizer, AutoModel

MODEL_NAMES = [
    # 'google/embeddinggemma-300m',
    os.path.abspath('../model/gemma300-vi-trimmed'),
]

for model_name in MODEL_NAMES:
    label = 'original' if model_name == MODEL_NAMES[0] else 'trimmed'
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    total = sum(p.numel() for p in model.parameters())
    embed = model.get_input_embeddings().weight.numel()
    print(f"[{label}] vocab: {tok.vocab_size:,} | total: {total/1e6:.1f}M | embed: {embed/1e6:.1f}M | other: {(total-embed)/1e6:.1f}M")

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/Users/minh/Code/Project/lm-vocab-trimmer/model/gemma300-vi-trimmed'. Use `repo_type` argument if needed.

In [6]:
import os
from sentence_transformers import SentenceTransformer
google_model = SentenceTransformer("google/embeddinggemma-300m")
hehe_model = SentenceTransformer(os.path.abspath('../model/gemma300-vi-trimmed-token'))

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

In [7]:
google_model

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'Gemma3TextModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Dense({'in_features': 768, 'out_features': 3072, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity', 'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
  (3): Dense({'in_features': 3072, 'out_features': 768, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity', 'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
  (4): Normalize({})
)

In [8]:
hehe_model

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'Gemma3TextModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Dense({'in_features': 768, 'out_features': 3072, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity', 'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
  (3): Dense({'in_features': 3072, 'out_features': 768, 'bias': False, 'activation_function': 'torch.nn.modules.linear.Identity', 'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
  (4): Normalize({})
)